# True Parametric PINN: train on (phi1, phi2), predict unseen in-range case

This notebook trains one single model with inputs `(t, phi1, phi2)` and predicts a new force case inside the training range.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'parametric_PINN'))

from true_parametric_pinn import (
    ParametricModelConfig,
    TrueParametricPINN,
    build_parametric_training_data,
)

In [ ]:
# ---------------- USER SETTINGS ----------------
cfg = ParametricModelConfig(
    n_dof=20,
    m_x=1.0,
    k=1.0,
    c=0.0,
    seg_window=1.0,
    layers=(3, 64, 64, 20),
    beta_ic=1.0,
    beta_ode=1.0,
    lr=1e-3,
)

# Training force cases (example 4 cases)
phi_train = [
    (0.10, 0.5),
    (0.10, 1.0),
    (0.10, 2.0),
    (0.30, 1.0),
]

n_collocation_t = 400
n_iter = 3000
print_every = 300

In [ ]:
# Build parametric training dataset
X_col, X_ic, x_ic, xt_ic = build_parametric_training_data(
    phi_cases=phi_train,
    n_collocation_t=n_collocation_t,
    seg_window=cfg.seg_window,
    n_dof=cfg.n_dof,
)

print('X_col shape:', X_col.shape)
print('X_ic shape :', X_ic.shape)

In [ ]:
# Train one single true parametric model
model = TrueParametricPINN(cfg)
model.train(X_col, X_ic, x_ic, xt_ic, n_iter=n_iter, print_every=print_every)

In [ ]:
# Predict a NEW unseen case inside training range
phi1_new = 0.20
phi2_new = 1.20

t_test = np.linspace(0, cfg.seg_window, 500)[:, None]
x_pred, xt_pred = model.predict(t_test, phi1=phi1_new, phi2=phi2_new)

print('Prediction shape:', x_pred.shape)

In [ ]:
# Quick visualization (last DOF)
plt.figure(figsize=(8, 4))
plt.plot(t_test[:, 0], x_pred[:, -1], lw=2)
plt.xlabel('t')
plt.ylabel('x_last')
plt.title(f'True parametric PINN prediction at phi1={phi1_new}, phi2={phi2_new}')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()